# A character-level LSTM that writes Shakespeare

Train an LSTM to predict the next character of Shakespeare's plays, then let it write. In the browser lab you saw why LSTMs keep gradients alive; here you'll see what that buys you. About 10 minutes on a GPU.

**How to use this notebook:** run the cells from top to bottom (Shift+Enter). In each step, replace every `___` in the code cell, run it, then run the **Check** cell under it. Stuck? Expand the hidden **Solution** cell (click the ▸ arrow). For speed, turn on a GPU first: *Runtime → Change runtime type → T4 GPU*.

## Setup
Run this once.

In [ ]:
import os, urllib.request, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
TEST = os.environ.get("NB_TEST") == "1"          # used only by the automatic tests; ignore it
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
if TEST:
    text = "To be, or not to be, that is the question.\nWhether tis nobler in the mind to suffer\n" * 200
else:
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = urllib.request.urlopen(url).read().decode("utf-8")
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
print(len(text), "characters,", V, "distinct")

## Step 1: Make training batches

Pick random starting points. The input is a chunk of L characters; the target is the same chunk shifted one character to the right (the next character at every position).

In [ ]:
def get_batch(L=100, B=64):
    starts = torch.randint(0, len(data) - L - 1, (B,))
    x = torch.stack([data[s:s + L] for s in starts])
    y = torch.stack([data[___] for s in starts])
    return x.to(device), y.to(device)

x, y = get_batch()
print(x.shape, y.shape)

In [ ]:
# Check: run me after the cell above
x, y = get_batch(L=20, B=4)
assert x.shape == y.shape == (4, 20) and torch.equal(x[:, 1:], y[:, :-1]), "y must be x shifted left by one"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
def get_batch(L=100, B=64):
    starts = torch.randint(0, len(data) - L - 1, (B,))
    x = torch.stack([data[s:s + L] for s in starts])
    y = torch.stack([data[s + 1:s + L + 1] for s in starts])
    return x.to(device), y.to(device)

x, y = get_batch()
print(x.shape, y.shape)

## Step 2: The model

Embedding (each character becomes a 64-number vector) → LSTM with 256 hidden units → linear layer back to one score per character.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, V, emb=64, hidden=256, cell=nn.LSTM):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.rnn = cell(___, ___, batch_first=True)
        self.out = nn.Linear(hidden, V)
    def forward(self, x, state=None):
        h, state = self.rnn(self.emb(x), state)
        return self.out(h), state

model = CharLSTM(V).to(device)
logits, _ = model(x)
print(logits.shape)

In [ ]:
# Check: run me after the cell above
assert model(x)[0].shape == (x.shape[0], x.shape[1], V)
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
class CharLSTM(nn.Module):
    def __init__(self, V, emb=64, hidden=256, cell=nn.LSTM):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.rnn = cell(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, V)
    def forward(self, x, state=None):
        h, state = self.rnn(self.emb(x), state)
        return self.out(h), state

model = CharLSTM(V).to(device)
logits, _ = model(x)
print(logits.shape)

## Step 3: Train with gradient clipping

Recurrent networks occasionally produce huge gradients. Clipping the gradient norm to 1.0 before each step keeps training stable.

In [ ]:
STEPS = 30 if TEST else 3000
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for step in range(STEPS):
    x, y = get_batch()
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(___, 1.0)
    opt.step(); losses.append(loss.item())
    if step % 500 == 0:
        print(step, round(loss.item(), 3))
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss")

In [ ]:
# Check: run me after the cell above
assert np.mean(losses[-10:]) < np.mean(losses[:10]), "Loss should fall"
if not TEST:
    assert np.mean(losses[-100:]) < 1.7, "Aim for a loss below 1.7"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
STEPS = 30 if TEST else 3000
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for step in range(STEPS):
    x, y = get_batch()
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); losses.append(loss.item())
    if step % 500 == 0:
        print(step, round(loss.item(), 3))
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss")

## Step 4: Let it write

Feed a prompt, then repeatedly sample the next character from softmax(logits / temperature). Low temperature is safe and repetitive; high temperature is creative and messy. Try 0.5, 0.8 and 1.2.

In [ ]:
def generate(prompt="ROMEO:", n=400, temperature=0.8):
    model.eval()
    x = torch.tensor([[stoi[c] for c in prompt if c in stoi] or [0]], dtype=torch.long, device=device)
    out, state = prompt, None
    with torch.no_grad():
        logits, state = model(x, state)
        for _ in range(n):
            probs = F.softmax(___, dim=-1)
            nxt = torch.multinomial(probs, 1)
            out += itos[nxt.item()]
            logits, state = model(nxt.view(1, 1), state)
    model.train()
    return out

print(generate())

In [ ]:
# Check: run me after the cell above
g = generate("To", n=50)
assert isinstance(g, str) and len(g) >= 50
print("Correct! Try swapping nn.LSTM for nn.RNN (cell=nn.RNN) and compare the loss.")

In [ ]:
#@title Solution (click ▸ to show)
def generate(prompt="ROMEO:", n=400, temperature=0.8):
    model.eval()
    x = torch.tensor([[stoi[c] for c in prompt if c in stoi] or [0]], dtype=torch.long, device=device)
    out, state = prompt, None
    with torch.no_grad():
        logits, state = model(x, state)
        for _ in range(n):
            probs = F.softmax(logits[0, -1] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1)
            out += itos[nxt.item()]
            logits, state = model(nxt.view(1, 1), state)
    model.train()
    return out

print(generate())